We pull 250 ranked solo-queue games from a high-elo KR Lulu one-trick via the Riot Match-V5 API. Each match is cached locally as JSON. From the match details and timeline events we extract: the lane opponent, whether the game was won, the rune page, starting items (first 60s of timeline events), and end-of-game core items.

In [1]:
from api import RiotAPIClient
from transform import build_dataframes
from report import aggregate_data

client = RiotAPIClient(api_key)
puuid = client.get_puuid(game_name, tag_line)
match_ids = client.get_match_ids(puuid, count=250, queue=420)

matches = {mid: client.get_match(mid) for mid in match_ids}
timelines = {mid: client.get_timeline(mid) for mid in match_ids}

matches_df, items_df, runes_df = build_dataframes(
    matches, timelines, puuid, champion_name="Lulu"
)
agg = aggregate_data(matches_df, items_df, runes_df)

qualifying = {k: v for k, v in agg.items() if v['total'] >= 3}
print(f"{len(match_ids)} ranked matches fetched (patches 15.24 \u2013 16.3)")
print(f"{len(matches_df)} Lulu games with a recognised lane opponent")
print(f"{len(qualifying)} distinct matchups (3+ games each)")

250 ranked matches fetched (patches 15.24 – 16.3)
164 Lulu games with a recognised lane opponent
26 distinct matchups (3+ games each)


The pipeline normalises everything into three DataFrames — one row per match, one per item event, one per rune selection — then aggregates per lane opponent.

In [2]:
summary.head(12)

,Games,Win Rate,Top Keystone,Boots
Fiora,5,100%,Electrocute,Swiftness
Dr. Mundo,4,100%,Electrocute,Swiftness
Darius,3,100%,Phase Rush,Swiftness
Anivia,3,100%,Electrocute,—
Vladimir,6,83%,Electrocute,Swiftness
Jax,10,80%,Electrocute,Steelcaps
Nautilus,4,75%,Summon Aery,Lucidity
Aurora,11,73%,Electrocute,Merc Treads
Renekton,10,70%,Electrocute,Steelcaps
Rumble,13,69%,Electrocute,Swiftness / Merc


The interesting pattern: this player runs Phase Rush *only* into Darius — a matchup where most players take Electrocute or Aery. Against Nautilus (likely a support-turned-top queue game) they switch to Summon Aery entirely, paired with support items like Dream Maker and Shurelya's. That's not a build you'd find on any guide site.